In [6]:
# %%
import numpy as np
import pandas as pd
from sklearn.preprocessing import RobustScaler
from sklearn.cluster import KMeans
from sklearn.preprocessing import TargetEncoder
from pathlib import Path

path_to_dataset = Path.cwd().parent / 'dataset'

# load both
train_df = pd.read_csv(path_to_dataset / 'train.csv')
test_df  = pd.read_csv(path_to_dataset / 'test.csv')

# keep original test id for submission
test_ids = test_df['id']



In [7]:
# %%
# TASK 1: DROP SAME COLUMNS
# test has no status_group so remove it from the drop list if needed

cols_to_drop = ['id', 'recorded_by', 'num_private', 'scheme_name', 'wpt_name', 'subvillage',
                'ward', 'date_recorded', 'region_code', 'district_code', 'quantity_group',
                'payment_type', 'quality_group', 'source_type', 'waterpoint_type_group',
                'extraction_type_group', 'management_group', 'public_meeting', 'scheme_management']

# only drop columns that exist in test
cols_to_drop_test = [c for c in cols_to_drop if c in test_df.columns]
test_df = test_df.drop(columns=cols_to_drop_test)



In [8]:
# %%
# TASK 2: MISSING VALUES

# replace zeros with NaN
test_df.loc[test_df['amount_tsh'] == 0, 'amount_tsh']             = np.nan
test_df.loc[test_df['construction_year'] == 0, 'construction_year'] = np.nan
test_df.loc[test_df['population'] == 0, 'population']             = np.nan
test_df.loc[test_df['longitude'] == 0, 'longitude']               = np.nan
test_df.loc[test_df['gps_height'] <= 0, 'gps_height']             = np.nan

# fill funder and installer NaN with Unknown
test_df['funder']    = test_df['funder'].fillna('Unknown')
test_df['installer'] = test_df['installer'].fillna('Unknown')

# fill permit with mode FROM TRAIN (not test)
train_permit_mode = train_df['permit'].mode()[0]
test_df['permit'] = test_df['permit'].fillna(train_permit_mode)

# fill with median grouped by region — using TRAIN medians
for col in ['amount_tsh', 'population', 'gps_height']:
    train_medians = train_df[train_df[col] > 0].groupby('region')[col].median()
    test_df[col]  = test_df[col].fillna(test_df['region'].map(train_medians))

# longitude grouped by extraction_type — using TRAIN medians
train_lon_medians = train_df[train_df['longitude'] > 0].groupby('extraction_type')['longitude'].median()
test_df['longitude'] = test_df['longitude'].fillna(test_df['extraction_type'].map(train_lon_medians))

# construction_year fill with 1986 (same as train)
test_df['construction_year'] = test_df['construction_year'].fillna(1986)

# safety fill with train global medians for anything still missing
for col in ['amount_tsh', 'population', 'gps_height', 'longitude']:
    train_global_median = train_df[train_df[col] > 0][col].median()
    test_df[col] = test_df[col].fillna(train_global_median)

print("Missing after imputation:")
print(test_df.isnull().sum()[test_df.isnull().sum() > 0])



Missing after imputation:
Series([], dtype: int64)


In [9]:
# %%
# TASK 3: ENCODING + SCALING

# --- frequency encoding for funder, installer, lga ---
# IMPORTANT: use TRAIN frequencies, not test
# because test might have categories not seen in train

percentage_funder    = round(train_df['funder'].value_counts() / len(train_df) * 100, 2)
percentage_installer = round(train_df['installer'].value_counts() / len(train_df) * 100, 2)
percentage_lga       = round(train_df['lga'].value_counts() / len(train_df) * 100, 2)

# clean typos first (same as train)
test_df['installer'] = test_df['installer'].str.strip().str.lower().str.title()
test_df['funder']    = test_df['funder'].str.strip().str.lower().str.title()

test_df['funder_frequency']    = test_df['funder'].map(percentage_funder)
test_df['installer_frequency'] = test_df['installer'].map(percentage_installer)
test_df['lga_frequency']       = test_df['lga'].map(percentage_lga)

# unseen categories in test will be NaN after map → fill with 0
test_df['funder_frequency']    = test_df['funder_frequency'].fillna(0)
test_df['installer_frequency'] = test_df['installer_frequency'].fillna(0)
test_df['lga_frequency']       = test_df['lga_frequency'].fillna(0)

test_df = test_df.drop(columns=['funder', 'installer', 'lga'])

# --- one-hot encode same columns as train ---
cat_cols = [c for c in test_df.select_dtypes(include='object').columns.tolist()
            if c not in ['funder', 'installer', 'lga', 'source']]

test_df = pd.get_dummies(test_df, columns=cat_cols, drop_first=False)

bool_cols = test_df.select_dtypes(include='bool').columns
test_df[bool_cols] = test_df[bool_cols].astype(int)

# --- align test columns with train ---
# load preprocessed train to get exact columns
preprocessed_train = pd.read_csv(path_to_dataset / 'preprocessed_train.csv')
train_cols = [c for c in preprocessed_train.columns if c != 'status_group']

# add missing columns to test as 0
for col in train_cols:
    if col not in test_df.columns:
        test_df[col] = 0

# remove extra columns test has that train doesn't
test_df = test_df[[c for c in train_cols if c in test_df.columns]]

print("Shape after encoding:", test_df.shape)
print("Train shape:", preprocessed_train.shape)

# %%
# --- log1p same features ---
test_df['amount_tsh'] = np.log1p(test_df['amount_tsh'])
test_df['population'] = np.log1p(test_df['population'])

# --- extract year and month from original test date ---
test_df['year_recorded']  = pd.to_datetime(pd.read_csv(path_to_dataset / 'test.csv')['date_recorded']).dt.year
test_df['month_recorded'] = pd.to_datetime(pd.read_csv(path_to_dataset / 'test.csv')['date_recorded']).dt.month

# --- scale using TRAIN scaler ---
# refit scaler on train (same columns)
num_cols = ['amount_tsh', 'gps_height', 'longitude', 'latitude', 'population']

train_preprocessed_copy = pd.read_csv(path_to_dataset / 'preprocessed_train.csv')
scaler = RobustScaler()
scaler.fit(train_preprocessed_copy[num_cols])  # fit on train only
test_df[num_cols] = scaler.transform(test_df[num_cols])  # transform test



C:\Users\abdul\AppData\Local\Temp\ipykernel_7836\2526979776.py:28: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = [c for c in test_df.select_dtypes(include='object').columns.tolist()
C:\Users\abdul\AppData\Local\Temp\ipykernel_7836\2526979776.py:44: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_df[col] = 0


Shape after encoding: (14850, 113)
Train shape: (59400, 114)


In [11]:
# %%
# TASK 4: FEATURE ENGINEERING (same as train)

# --- reload originals cleanly ---
train_original = pd.read_csv(path_to_dataset / 'train.csv')
test_original  = pd.read_csv(path_to_dataset / 'test.csv')
train_saved    = pd.read_csv(path_to_dataset / 'preprocessed_train.csv')

# --- geo_cluster: fit kmeans on train, predict on test ---
kmeans = KMeans(n_clusters=12, random_state=42)
kmeans.fit(train_saved[['longitude', 'latitude']])
test_df['geo_cluster'] = kmeans.predict(test_df[['longitude', 'latitude']])

# --- pump_age ---
test_df['pump_age'] = test_df['year_recorded'] - test_df['construction_year']

# --- geo_cluster_source ---
# rebuild on train side
train_geo_source = (
    train_saved['geo_cluster'].astype(str) + '_' + train_original['source'].values
)
# build on test side
test_geo_source = (
    test_df['geo_cluster'].astype(str) + '_' + test_original['source'].values
)

# target encode: fit on train, transform on test
target_encoder = TargetEncoder()
target_encoder.fit(
    train_geo_source.values.reshape(-1, 1),
    train_saved['status_group'].values
)
test_df['geo_cluster_source'] = target_encoder.transform(
    test_geo_source.values.reshape(-1, 1)
)

# %%
# final column alignment — make test match train exactly
train_cols = [c for c in train_saved.columns if c != 'status_group']

for col in train_cols:
    if col not in test_df.columns:
        test_df[col] = 0

test_df = test_df[train_cols]

print("Final test shape :", test_df.shape)
print("Final train shape:", len(train_cols))

# %%
test_df.to_csv(path_to_dataset / 'preprocessed_test.csv', index=False)
pd.DataFrame({'id': test_ids}).to_csv(path_to_dataset / 'test_ids.csv', index=False)
print("Done")

Final test shape : (14850, 113)
Final train shape: 113
Done
